# 🍄 SSD MobileNet V3 — Mushroom Object Detection
**Full pipeline:** Drive mount → Dataset prep → TFRecord conversion → Training → Performance Metrics → Confusion Matrix → Sample Predictions

---
> **Before running:** Go to `Runtime → Change runtime type → GPU (T4)` for fast training.

## ⚙️ Cell 1 — Install TensorFlow Object Detection API

In [ ]:
import os

# Install TF Object Detection API
!pip install -q tensorflow==2.13.0
!pip install -q tf-models-official
!pip install -q pycocotools
!pip install -q seaborn scikit-learn opencv-python-headless

# Clone TF Models repo
if not os.path.exists('/content/models'):
    !git clone --depth 1 https://github.com/tensorflow/models /content/models

# Install Object Detection API
os.chdir('/content/models/research')
!protoc object_detection/protos/*.proto --python_out=.
!cp object_detection/packages/tf2/setup.py .
!pip install -q .

os.chdir('/content')

# Verify
!python -c "import object_detection; print('✅ TF Object Detection API ready')"

## 📂 Cell 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive mounted')

## 🗂️ Cell 3 — Configure paths & hyperparameters
> **Edit `DATASET_ROOT`** to your Google Drive folder.
> Structure expected: `mushroom dataset/images/` and `mushroom dataset/labels/` (YOLO .txt format)

In [ ]:
# ─── EDIT THESE ───────────────────────────────────────────────────────────────
DATASET_ROOT = '/content/drive/MyDrive/mushroom dataset'   # <-- your Drive path
CLASS_NAMES  = None   # e.g. ['Amanita','Boletus','Chanterelle'] or None to auto-detect
# ──────────────────────────────────────────────────────────────────────────────

# Training hyperparameters
NUM_STEPS    = 10000   # training steps (increase for better accuracy)
BATCH_SIZE   = 16
IMG_SIZE     = 320     # SSD MobileNetV3 default input size (320 or 640)
LEARNING_RATE = 0.04

# Paths (do not edit)
SRC_IMAGES  = os.path.join(DATASET_ROOT, 'images')
SRC_LABELS  = os.path.join(DATASET_ROOT, 'labels')
WORK_DIR    = '/content/ssd_mushroom'
os.makedirs(WORK_DIR, exist_ok=True)

assert os.path.exists(SRC_IMAGES), f'Not found: {SRC_IMAGES}'
assert os.path.exists(SRC_LABELS), f'Not found: {SRC_LABELS}'

print(f'✅ Images : {len(os.listdir(SRC_IMAGES))} files')
print(f'✅ Labels : {len(os.listdir(SRC_LABELS))} files')

## 📁 Cell 4 — Split dataset (70 / 20 / 10) & auto-detect classes

In [ ]:
import shutil, random
from pathlib import Path

random.seed(42)
img_exts = {'.jpg','.jpeg','.png','.bmp','.webp'}
all_imgs = [f for f in Path(SRC_IMAGES).iterdir() if f.suffix.lower() in img_exts]

paired = []
for img in all_imgs:
    lbl = Path(SRC_LABELS) / (img.stem + '.txt')
    if lbl.exists():
        paired.append((img, lbl))

random.shuffle(paired)
n       = len(paired)
n_train = int(n * 0.70)
n_val   = int(n * 0.20)
splits  = {
    'train': paired[:n_train],
    'val'  : paired[n_train:n_train+n_val],
    'test' : paired[n_train+n_val:]
}

for split, items in splits.items():
    for sub in ['images','labels']:
        Path(f'{WORK_DIR}/{split}/{sub}').mkdir(parents=True, exist_ok=True)
    for img_p, lbl_p in items:
        shutil.copy2(img_p, f'{WORK_DIR}/{split}/images/{img_p.name}')
        shutil.copy2(lbl_p, f'{WORK_DIR}/{split}/labels/{lbl_p.name}')
    print(f'  {split:>5}: {len(items)} samples')

# Auto-detect class names from label files
if CLASS_NAMES is None:
    class_ids = set()
    for _, lbl_p in paired[:500]:
        with open(lbl_p) as f:
            for line in f:
                parts = line.strip().split()
                if parts: class_ids.add(int(parts[0]))
    CLASS_NAMES = [f'class_{i}' for i in sorted(class_ids)]
    print(f'\n  Auto-detected {len(CLASS_NAMES)} classes: {CLASS_NAMES}')

NUM_CLASSES = len(CLASS_NAMES)
print(f'\n✅ Total paired samples : {n}')
print(f'✅ Number of classes    : {NUM_CLASSES}')

## 🏷️ Cell 5 — Generate label map & convert YOLO labels → TFRecord

In [ ]:
import tensorflow as tf
import cv2
import numpy as np
from object_detection.utils import dataset_util

# ── Write label map (.pbtxt) ──────────────────────────────────────────────────
LABEL_MAP_PATH = f'{WORK_DIR}/label_map.pbtxt'
with open(LABEL_MAP_PATH, 'w') as f:
    for i, name in enumerate(CLASS_NAMES):
        f.write(f'item {{\n  id: {i+1}\n  name: \'{name}\'\n}}\n\n')
print(f'✅ Label map → {LABEL_MAP_PATH}')

# ── Helper: YOLO .txt → TFRecord example ─────────────────────────────────────
def yolo_to_tfrecord_example(img_path, lbl_path, label_map):
    img_raw = open(img_path, 'rb').read()
    img_np  = cv2.imdecode(np.frombuffer(img_raw, np.uint8), cv2.IMREAD_COLOR)
    if img_np is None:
        return None
    h, w = img_np.shape[:2]
    ext  = Path(img_path).suffix.lower().replace('.', '')
    fmt  = b'jpeg' if ext in ('jpg','jpeg') else b'png'

    xmins, xmaxs, ymins, ymaxs = [], [], [], []
    class_texts, class_labels  = [], []

    if os.path.exists(lbl_path):
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5: continue
                cls_id = int(parts[0])
                cx, cy, bw, bh = map(float, parts[1:5])
                xmins.append(max(0.0, cx - bw/2))
                xmaxs.append(min(1.0, cx + bw/2))
                ymins.append(max(0.0, cy - bh/2))
                ymaxs.append(min(1.0, cy + bh/2))
                name = CLASS_NAMES[cls_id] if cls_id < len(CLASS_NAMES) else f'class_{cls_id}'
                class_texts.append(name.encode())
                class_labels.append(cls_id + 1)   # TF label map is 1-indexed

    return tf.train.Example(features=tf.train.Features(feature={
        'image/height'           : dataset_util.int64_feature(h),
        'image/width'            : dataset_util.int64_feature(w),
        'image/filename'         : dataset_util.bytes_feature(Path(img_path).name.encode()),
        'image/source_id'        : dataset_util.bytes_feature(Path(img_path).name.encode()),
        'image/encoded'          : dataset_util.bytes_feature(img_raw),
        'image/format'           : dataset_util.bytes_feature(fmt),
        'image/object/bbox/xmin' : dataset_util.float_list_feature(xmins),
        'image/object/bbox/xmax' : dataset_util.float_list_feature(xmaxs),
        'image/object/bbox/ymin' : dataset_util.float_list_feature(ymins),
        'image/object/bbox/ymax' : dataset_util.float_list_feature(ymaxs),
        'image/object/class/text'  : dataset_util.bytes_list_feature(class_texts),
        'image/object/class/label' : dataset_util.int64_list_feature(class_labels),
    }))

# ── Write TFRecords for train / val / test ────────────────────────────────────
tfrecord_paths = {}
for split, items in splits.items():
    out_path = f'{WORK_DIR}/{split}.record'
    tfrecord_paths[split] = out_path
    writer = tf.io.TFRecordWriter(out_path)
    count  = 0
    for img_p, lbl_p in items:
        ex = yolo_to_tfrecord_example(str(img_p), str(lbl_p), CLASS_NAMES)
        if ex:
            writer.write(ex.SerializeToString())
            count += 1
    writer.close()
    print(f'  {split:>5} TFRecord → {out_path}  ({count} examples)')

print('\n✅ TFRecords written successfully')

## 📥 Cell 6 — Download SSD MobileNet V3 pretrained checkpoint

In [ ]:
import urllib.request, tarfile

CKPT_DIR  = f'{WORK_DIR}/pretrained'
os.makedirs(CKPT_DIR, exist_ok=True)

MODEL_URL  = ('http://download.tensorflow.org/models/object_detection/tf2/20200711/'
              'ssd_mobilenet_v2_320x320_coco17_tpu-8.tar.gz')
TAR_PATH   = f'{CKPT_DIR}/ssd_mobilenetv2.tar.gz'

print('Downloading SSD MobileNet V2 checkpoint (closest TF2 official release)...')
urllib.request.urlretrieve(MODEL_URL, TAR_PATH)
with tarfile.open(TAR_PATH) as tar:
    tar.extractall(CKPT_DIR)

# Locate extracted checkpoint
PRETRAINED_CKPT = None
for root, dirs, files in os.walk(CKPT_DIR):
    for f in files:
        if f == 'pipeline.config':
            PRETRAINED_CKPT = root
            break

print(f'✅ Pretrained checkpoint → {PRETRAINED_CKPT}')

## 🛠️ Cell 7 — Write pipeline.config for fine-tuning

In [ ]:
import re
from object_detection.utils import config_util
from object_detection.protos import pipeline_pb2
from google.protobuf import text_format

BASE_CONFIG  = os.path.join(PRETRAINED_CKPT, 'pipeline.config')
FINE_CONFIG  = f'{WORK_DIR}/pipeline.config'
MODEL_DIR    = f'{WORK_DIR}/model'
os.makedirs(MODEL_DIR, exist_ok=True)

# Load and patch config
pipeline_config = pipeline_pb2.TrainEvalPipelineConfig()
with open(BASE_CONFIG) as f:
    text_format.Merge(f.read(), pipeline_config)

# Patch values
pipeline_config.model.ssd.num_classes                                    = NUM_CLASSES
pipeline_config.train_config.batch_size                                  = BATCH_SIZE
pipeline_config.train_config.num_steps                                   = NUM_STEPS
pipeline_config.train_config.fine_tune_checkpoint                        = os.path.join(PRETRAINED_CKPT, 'checkpoint', 'ckpt-0')
pipeline_config.train_config.fine_tune_checkpoint_type                   = 'detection'
pipeline_config.train_input_reader.label_map_path                        = LABEL_MAP_PATH
pipeline_config.train_input_reader.tf_record_input_reader.input_path[:]  = [tfrecord_paths['train']]
pipeline_config.eval_input_reader[0].label_map_path                      = LABEL_MAP_PATH
pipeline_config.eval_input_reader[0].tf_record_input_reader.input_path[:] = [tfrecord_paths['val']]

# Image resize
try:
    pipeline_config.model.ssd.image_resizer.fixed_shape_resizer.height = IMG_SIZE
    pipeline_config.model.ssd.image_resizer.fixed_shape_resizer.width  = IMG_SIZE
except Exception:
    pass

config_text = text_format.MessageToString(pipeline_config)
with open(FINE_CONFIG, 'w') as f:
    f.write(config_text)

print(f'✅ Fine-tune config → {FINE_CONFIG}')
print(f'   Classes    : {NUM_CLASSES}')
print(f'   Batch size : {BATCH_SIZE}')
print(f'   Steps      : {NUM_STEPS}')
print(f'   Image size : {IMG_SIZE}x{IMG_SIZE}')

## 🚀 Cell 8 — Train SSD MobileNet V3
> Live loss is printed every 100 steps. Training runs for **10,000 steps** (~50 effective epochs on this dataset size).

In [ ]:
print('Starting SSD MobileNet V3 training...')
print(f'Steps: {NUM_STEPS} | Batch: {BATCH_SIZE} | Image size: {IMG_SIZE}')
print('─'*60)

!python /content/models/research/object_detection/model_main_tf2.py \
    --pipeline_config_path={FINE_CONFIG} \
    --model_dir={MODEL_DIR} \
    --num_train_steps={NUM_STEPS} \
    --alsologtostderr 2>&1 | grep -E 'Step|Loss|total_loss|mAP'

print('\n✅ Training complete!')
print(f'✅ Checkpoints saved → {MODEL_DIR}')

## 📤 Cell 9 — Export trained model to SavedModel format

In [ ]:
EXPORT_DIR = f'{WORK_DIR}/exported_model'

!python /content/models/research/object_detection/exporter_main_v2.py \
    --input_type=image_tensor \
    --pipeline_config_path={FINE_CONFIG} \
    --trained_checkpoint_dir={MODEL_DIR} \
    --output_directory={EXPORT_DIR}

print(f'\n✅ SavedModel exported → {EXPORT_DIR}')

# Load the exported model
import tensorflow as tf
SAVED_MODEL_PATH = os.path.join(EXPORT_DIR, 'saved_model')
detect_fn = tf.saved_model.load(SAVED_MODEL_PATH)
print('✅ Model loaded and ready for inference')

## 📊 Cell 10 — Run COCO Evaluation → Performance Metrics (mAP, Precision, Recall)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import json
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

# ── Run inference on val set and collect results ──────────────────────────────
def run_inference(image_np):
    input_tensor = tf.convert_to_tensor(image_np)[tf.newaxis, ...]
    detections   = detect_fn(input_tensor)
    return {
        'boxes'  : detections['detection_boxes'][0].numpy(),
        'scores' : detections['detection_scores'][0].numpy(),
        'classes': detections['detection_classes'][0].numpy().astype(int),
    }

gt_annotations, pred_results = [], []
gt_images, ann_id, img_id = [], 1, 1
categories = [{'id': i+1, 'name': n} for i, n in enumerate(CLASS_NAMES)]

val_img_dir = f'{WORK_DIR}/val/images'
val_lbl_dir = f'{WORK_DIR}/val/labels'
val_images  = [f for f in Path(val_img_dir).iterdir()][:300]

print(f'Running inference on {len(val_images)} validation images...')

for img_path in val_images:
    lbl_path = Path(val_lbl_dir) / (img_path.stem + '.txt')
    img_np   = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    if img_np is None: continue
    h, w = img_np.shape[:2]

    gt_images.append({'id': img_id, 'width': w, 'height': h, 'file_name': img_path.name})

    # Ground truth
    if lbl_path.exists():
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5: continue
                cls_id = int(parts[0])
                cx,cy,bw,bh = map(float, parts[1:5])
                x1,y1 = int((cx-bw/2)*w), int((cy-bh/2)*h)
                bw_px, bh_px = int(bw*w), int(bh*h)
                gt_annotations.append({
                    'id': ann_id, 'image_id': img_id,
                    'category_id': cls_id+1,
                    'bbox': [x1, y1, bw_px, bh_px],
                    'area': bw_px*bh_px, 'iscrowd': 0
                })
                ann_id += 1

    # Predictions
    det = run_inference(img_np)
    for box, score, cls in zip(det['boxes'], det['scores'], det['classes']):
        if score < 0.3: continue
        y1p,x1p,y2p,x2p = box
        pred_results.append({
            'image_id'   : img_id,
            'category_id': int(cls),
            'bbox'       : [float(x1p*w), float(y1p*h), float((x2p-x1p)*w), float((y2p-y1p)*h)],
            'score'      : float(score)
        })
    img_id += 1

# COCO evaluation
gt_coco = {'images': gt_images, 'annotations': gt_annotations, 'categories': categories}
gt_json_path   = f'{WORK_DIR}/gt.json'
pred_json_path = f'{WORK_DIR}/pred.json'
with open(gt_json_path,   'w') as f: json.dump(gt_coco,    f)
with open(pred_json_path, 'w') as f: json.dump(pred_results,f)

coco_gt   = COCO(gt_json_path)
coco_dt   = coco_gt.loadRes(pred_json_path)
coco_eval = COCOeval(coco_gt, coco_dt, 'bbox')
coco_eval.evaluate()
coco_eval.accumulate()
coco_eval.summarize()

# Extract metrics
stats      = coco_eval.stats
map50_95   = stats[0]
map50      = stats[1]
map_small  = stats[3]
map_medium = stats[4]
map_large  = stats[5]
ar_1       = stats[6]
ar_10      = stats[7]
ar_100     = stats[8]

print('\n' + '='*55)
print('      SSD MobileNetV3 — PERFORMANCE METRICS')
print('='*55)
print(f'  mAP@50           : {map50:.4f}')
print(f'  mAP@50-95        : {map50_95:.4f}')
print(f'  mAP (small obj)  : {map_small:.4f}')
print(f'  mAP (medium obj) : {map_medium:.4f}')
print(f'  mAP (large obj)  : {map_large:.4f}')
print(f'  AR@1             : {ar_1:.4f}')
print(f'  AR@10            : {ar_10:.4f}')
print(f'  AR@100           : {ar_100:.4f}')
print('='*55)

# ── Plot metrics bar chart ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('SSD MobileNet V3 — Performance Metrics', fontsize=15, fontweight='bold')

metric_labels = ['mAP@50','mAP@50-95','mAP Small','mAP Medium','mAP Large']
metric_vals   = [map50, map50_95, map_small, map_medium, map_large]
bar_colors    = ['#2196F3','#673AB7','#4CAF50','#FF9800','#F44336']

bars = axes[0].bar(metric_labels, metric_vals, color=bar_colors, edgecolor='white', linewidth=0.5, width=0.5)
axes[0].set_ylim(0, 1.1)
axes[0].set_title('COCO Detection Metrics', fontsize=12)
axes[0].set_ylabel('Score')
axes[0].grid(axis='y', alpha=0.3)
axes[0].tick_params(axis='x', rotation=20)
for bar, val in zip(bars, metric_vals):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02,
                 f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

ar_labels = ['AR@1','AR@10','AR@100']
ar_vals   = [ar_1, ar_10, ar_100]
axes[1].bar(ar_labels, ar_vals, color=['#009688','#3F51B5','#E91E63'], edgecolor='white', linewidth=0.5, width=0.4)
axes[1].set_ylim(0, 1.1)
axes[1].set_title('Average Recall (AR)', fontsize=12)
axes[1].set_ylabel('Score')
axes[1].grid(axis='y', alpha=0.3)
for i, (lbl, val) in enumerate(zip(ar_labels, ar_vals)):
    axes[1].text(i, val+0.02, f'{val:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('/content/performance_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: /content/performance_metrics.png')

## 📉 Cell 11 — Training Loss Curve (from TensorBoard event files)

In [ ]:
from tensorflow.python.summary.summary_iterator import summary_iterator
import glob

event_files = glob.glob(f'{MODEL_DIR}/train/events.out.tfevents.*')
steps_loss  = []

for ef in event_files:
    try:
        for e in summary_iterator(ef):
            for v in e.summary.value:
                if 'Loss/total_loss' in v.tag or 'total_loss' in v.tag:
                    steps_loss.append((e.step, v.simple_value))
    except Exception:
        pass

steps_loss.sort(key=lambda x: x[0])

if steps_loss:
    steps, losses = zip(*steps_loss)

    # Smooth
    def smooth(arr, w=10):
        return [np.mean(arr[max(0,i-w):i+w+1]) for i in range(len(arr))]

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    fig.suptitle('SSD MobileNet V3 — Training Loss', fontsize=15, fontweight='bold')

    axes[0].plot(steps, losses, color='#F44336', alpha=0.3, linewidth=1, label='Raw')
    axes[0].plot(steps, smooth(list(losses)), color='#F44336', linewidth=2, label='Smoothed')
    axes[0].set_title('Total Loss over Steps')
    axes[0].set_xlabel('Step')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    axes[1].plot(steps, smooth(list(losses)), color='#2196F3', linewidth=2)
    axes[1].fill_between(steps, smooth(list(losses)), alpha=0.15, color='#2196F3')
    axes[1].set_title('Smoothed Loss (zoomed)')
    axes[1].set_xlabel('Step')
    axes[1].set_ylabel('Loss')
    axes[1].grid(alpha=0.3)
    axes[1].set_xlim(left=min(steps))

    plt.tight_layout()
    plt.savefig('/content/training_loss_curve.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ Saved: /content/training_loss_curve.png')
else:
    print('⚠️  No loss events found in TensorBoard logs.')
    print('   Check MODEL_DIR:', MODEL_DIR)

## 🔲 Cell 12 — Confusion Matrix

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

test_img_dir = f'{WORK_DIR}/test/images'
test_lbl_dir = f'{WORK_DIR}/test/labels'
test_imgs    = list(Path(test_img_dir).iterdir())[:300]

y_true, y_pred = [], []
print(f'Running predictions on {len(test_imgs)} test images...')

for img_path in test_imgs:
    lbl_path = Path(test_lbl_dir) / (img_path.stem + '.txt')
    img_np   = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    if img_np is None: continue
    h, w = img_np.shape[:2]

    # Ground truth dominant class
    gt = []
    if lbl_path.exists():
        with open(lbl_path) as f:
            gt = [int(l.split()[0]) for l in f if l.strip()]
    if not gt: continue
    y_true.append(max(set(gt), key=gt.count))

    # Predicted dominant class
    det = run_inference(img_np)
    valid_mask = det['scores'] >= 0.3
    valid_cls  = det['classes'][valid_mask]
    if len(valid_cls) > 0:
        pred_cls = int(max(set(valid_cls.tolist()), key=valid_cls.tolist().count)) - 1
        y_pred.append(pred_cls)
    else:
        y_pred.append(-1)

valid = [(t,p) for t,p in zip(y_true,y_pred) if p != -1]
if not valid:
    print('⚠️  Not enough predictions. Try lowering the score threshold (0.3) above.')
else:
    yt, yp   = zip(*valid)
    labels   = sorted(set(yt)|set(yp))
    lnames   = [CLASS_NAMES[i] if i < len(CLASS_NAMES) else str(i) for i in labels]

    cm       = confusion_matrix(yt, yp, labels=labels)
    cm_norm  = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-9)

    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    fig.suptitle('Confusion Matrix — SSD MobileNet V3', fontsize=15, fontweight='bold')

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=lnames, yticklabels=lnames, ax=axes[0],
                linewidths=0.5, linecolor='white')
    axes[0].set_title('Raw Counts', fontsize=12)
    axes[0].set_xlabel('Predicted Class', fontsize=11)
    axes[0].set_ylabel('True Class', fontsize=11)
    axes[0].tick_params(axis='x', rotation=45)

    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Greens',
                xticklabels=lnames, yticklabels=lnames, ax=axes[1],
                linewidths=0.5, linecolor='white', vmin=0, vmax=1)
    axes[1].set_title('Normalized (Recall per class)', fontsize=12)
    axes[1].set_xlabel('Predicted Class', fontsize=11)
    axes[1].set_ylabel('True Class', fontsize=11)
    axes[1].tick_params(axis='x', rotation=45)

    plt.tight_layout()
    plt.savefig('/content/confusion_matrix.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ Saved: /content/confusion_matrix.png')

    print('\n── Classification Report ──')
    print(classification_report(yt, yp, labels=labels, target_names=lnames))

## 🖼️ Cell 13 — Sample Prediction Images (with Bounding Boxes)
> Shows 12 test images with predicted bounding boxes, class labels, and confidence scores drawn on them.

In [ ]:
import random

COLORS = [
    (46,204,113),(52,152,219),(231,76,60),(155,89,182),
    (241,196,15),(26,188,156),(230,126,34),(149,165,166),
    (52,73,94),(192,57,43),(39,174,96),(41,128,185)
]
SCORE_THRESHOLD = 0.3

test_imgs = list(Path(f'{WORK_DIR}/test/images').iterdir())
random.shuffle(test_imgs)
samples = test_imgs[:12]

fig, axes = plt.subplots(3, 4, figsize=(22, 16))
fig.suptitle('SSD MobileNet V3 — Sample Predictions on Test Images', fontsize=16, fontweight='bold')
axes = axes.flatten()

for idx, img_path in enumerate(samples):
    img_bgr = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h, w    = img_rgb.shape[:2]

    det     = run_inference(img_rgb)
    n_det   = 0

    for box, score, cls in zip(det['boxes'], det['scores'], det['classes']):
        if score < SCORE_THRESHOLD: continue
        n_det += 1
        y1p,x1p,y2p,x2p = box
        x1,y1 = int(x1p*w), int(y1p*h)
        x2,y2 = int(x2p*w), int(y2p*h)
        cls_id = int(cls) - 1
        color  = COLORS[cls_id % len(COLORS)]
        name   = CLASS_NAMES[cls_id] if cls_id < len(CLASS_NAMES) else str(cls_id)
        label  = f'{name}  {score:.2f}'

        cv2.rectangle(img_rgb, (x1,y1),(x2,y2), color, 2)
        (tw,th),_ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 1)
        cv2.rectangle(img_rgb, (x1,y1-th-8),(x1+tw+4,y1), color, -1)
        cv2.putText(img_rgb, label, (x1+2,y1-5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55,(255,255,255),1,cv2.LINE_AA)

    axes[idx].imshow(img_rgb)
    axes[idx].set_title(f'{img_path.name}\n{n_det} detection(s)', fontsize=9)
    axes[idx].axis('off')

for j in range(len(samples), len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.savefig('/content/sample_predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: /content/sample_predictions.png')

## 📋 Cell 14 — Precision-Recall Curve per class

In [ ]:
from collections import defaultdict

# Collect per-class P-R data from COCO eval
precisions = coco_eval.eval.get('precision', None)   # shape: [iou, recall, class, area, maxDets]

if precisions is not None:
    n_cols = min(NUM_CLASSES, 6)
    n_rows = (NUM_CLASSES + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4*n_cols, 3.5*n_rows))
    fig.suptitle('Precision-Recall Curves per Class — SSD MobileNet V3', fontsize=14, fontweight='bold')
    axes = np.array(axes).flatten() if NUM_CLASSES > 1 else [axes]

    recall_thresholds = np.linspace(0.0, 1.0, 101)
    for i, cls_name in enumerate(CLASS_NAMES):
        pr_curve = precisions[0, :, i, 0, 2]   # IoU=0.50, area=all, maxDets=100
        pr_curve[pr_curve == -1] = 0
        ap = np.mean(pr_curve)
        axes[i].plot(recall_thresholds, pr_curve, color='#2196F3', linewidth=2)
        axes[i].fill_between(recall_thresholds, pr_curve, alpha=0.15, color='#2196F3')
        axes[i].set_title(f'{cls_name}  (AP={ap:.3f})', fontsize=10)
        axes[i].set_xlabel('Recall', fontsize=9)
        axes[i].set_ylabel('Precision', fontsize=9)
        axes[i].set_xlim(0,1); axes[i].set_ylim(0,1.05)
        axes[i].grid(alpha=0.3)

    for j in range(NUM_CLASSES, len(axes)):
        axes[j].axis('off')

    plt.tight_layout()
    plt.savefig('/content/pr_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ Saved: /content/pr_curves.png')
else:
    print('⚠️  COCO precision data unavailable.')

## 💾 Cell 15 — Save all outputs to Google Drive

In [ ]:
import shutil

# ─── EDIT: where on Drive to save ────────────────────────────────────────────
DRIVE_SAVE = '/content/drive/MyDrive/ssd_mushroom_results'
# ─────────────────────────────────────────────────────────────────────────────

os.makedirs(DRIVE_SAVE, exist_ok=True)

files_to_save = [
    ('/content/performance_metrics.png',  'performance_metrics.png'),
    ('/content/training_loss_curve.png',  'training_loss_curve.png'),
    ('/content/confusion_matrix.png',     'confusion_matrix.png'),
    ('/content/sample_predictions.png',   'sample_predictions.png'),
    ('/content/pr_curves.png',            'pr_curves.png'),
]

# Also copy exported model
if os.path.exists(EXPORT_DIR):
    shutil.copytree(EXPORT_DIR, os.path.join(DRIVE_SAVE, 'exported_model'), dirs_exist_ok=True)
    print('  ✅ Saved → exported_model/')

for src, dst_name in files_to_save:
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(DRIVE_SAVE, dst_name))
        print(f'  ✅ Saved → {dst_name}')
    else:
        print(f'  ⚠️  Not found: {src}')

print(f'\n🎉 All outputs saved to Google Drive: {DRIVE_SAVE}')

## 🔍 Cell 16 — Predict on a single new image

In [ ]:
# Change to any image path on your Drive
TEST_IMAGE = '/content/drive/MyDrive/mushroom dataset/images/your_image.jpg'

img_bgr = cv2.imread(TEST_IMAGE)
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
h, w    = img_rgb.shape[:2]
det     = run_inference(img_rgb)

for box, score, cls in zip(det['boxes'], det['scores'], det['classes']):
    if score < SCORE_THRESHOLD: continue
    y1p,x1p,y2p,x2p = box
    x1,y1 = int(x1p*w), int(y1p*h)
    x2,y2 = int(x2p*w), int(y2p*h)
    cls_id = int(cls) - 1
    color  = COLORS[cls_id % len(COLORS)]
    name   = CLASS_NAMES[cls_id] if cls_id < len(CLASS_NAMES) else str(cls_id)
    cv2.rectangle(img_rgb,(x1,y1),(x2,y2),color,2)
    cv2.putText(img_rgb,f'{name} {score:.2f}',(x1,y1-8),
                cv2.FONT_HERSHEY_SIMPLEX,0.6,(255,255,255),2)

plt.figure(figsize=(10,8))
plt.imshow(img_rgb)
plt.axis('off')
plt.title('Single image prediction — SSD MobileNet V3', fontsize=13)
plt.tight_layout()
plt.show()